In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid
from datetime import datetime, UTC

In [0]:
spark.sql("use catalog novacart_db")
spark.sql("create schema if not exists gold_schema")
gold_run_id = str(uuid.uuid4())

run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print("Current Gold Run ID:", gold_run_id)
print("Run Timestamp Folder:", run_ts_str)
     

In [0]:

spark.sql("""
          create table if not exists novacart_db.gold_schema.processing_control(
              layer string,
              entity_name string,
              last_processed_silver_run_id string,
              last_processed_silver_run_ts timestamp,
              rows_merged bigint,
              run_status string,
              gold_run_id string,
              updated_at timestamp
          )
          using delta
          """)

In [0]:
def upsert_to_gold(df_source, target_table, join_key):
    if spark.catalog.tableExists(target_table):
        dt = DeltaTable.forName(spark, target_table)
        (dt.alias("target")
         .merge(df_source.alias("source"), f"target.{join_key} = source.{join_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
        
    else:
        df_source.write.format("delta").saveAsTable(target_table)    

In [0]:
def get_last_processed_silver_ts(entity_name:str):
    ctrl= (
        spark.table("novacart_db.gold_schema.processing_control").filter(
            (F.col("layer") =='gold') &
            (F.col("entity_name") == entity_name) &
            (F.col("run_status") =='success')
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None
    return rows[0]["last_processed_silver_run_ts"]

In [0]:
def upser_gold_control(entity_name, last_processed_silver_run_id, last_processed_silver_run_ts, rows_merged):
    ctrl_df  = spark.createDataFrame(
        [(
            "gold",
            entity_name,
            last_processed_silver_run_id,
            last_processed_silver_run_ts,
            int(rows_merged),
            "success",
            gold_run_id,
            datetime.now(UTC)
        )],
        schema = """
            layer string,
            entity_name string,
            last_processed_silver_run_id string,
            last_processed_silver_run_ts timestamp,
            rows_merged bigint,
            run_status string,
            gold_run_id string,
            updated_at timestamp
            """
    )

    dt = DeltaTable.forName(spark, "novacart_db.gold_schema.processing_control")
    (dt.alias("t")
     .merge(ctrl_df.alias("s"), "t.layer = s.layer and t.entity_name = s.entity_name")
     .whenMatchedUpdate(set={
         "last_processed_silver_run_id": "s.last_processed_silver_run_id",
         "last_processed_silver_run_ts": "s.last_processed_silver_run_ts",
         "rows_merged": "s.rows_merged",
         "run_status": "s.run_status",
         "gold_run_id": "s.gold_run_id",
         "updated_at": "s.updated_at"
     })
     .whenNotMatchedInsertAll()
     .execute())

In [0]:
last_gold_ts = get_last_processed_silver_ts("orders_information")

print("Last Processed Silver Timestamp for Gold = ", last_gold_ts)

silver_orders_current = spark.read.table("novacart_db.silver_schema.orders_transformed")
silver_products_current = spark.read.table("novacart_db.silver_schema.products_transformed")
silver_payments_current = spark.read.table("novacart_db.silver_schema.payments_transformed")

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:
    changed_orders = silver_orders_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_products = silver_products_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(F.col("processed_at") > F.lit(last_gold_ts))

changed_order_count = changed_orders.count()
changed_product_count = changed_products.count()
changed_payment_count = changed_payments.count()

print(f"Number of changed orders = {changed_order_count}")
print(f"Number of changed products = {changed_product_count}")
print(f"Number of changed payments = {changed_payment_count}")

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products = (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"),
          F.col("p.product_id") == F.col("o.product_id"),
          "inner")
    .select(F.col("o.order_id")).distinct()
)

impacted_orders_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)

print("impacted_order_ids = ",impacted_orders_ids.count())
display(impacted_orders_ids.orderBy("order_id"))

In [0]:
impacted_orders = (
    silver_orders_current.alias("o")
    .join(impacted_orders_ids.alias("i"), "order_id", "inner")
)

gold_delta = (
    impacted_orders.alias("o")
    .join(
        silver_products_current.alias("p"),
        F.col("o.product_id") == F.col("p.product_id"),
        "inner"
    )
    .join(
        silver_payments_current.alias("py"),
        F.col("o.order_id") == F.col("py.order_id"),
        "inner"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("product_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("py.payment_id"),
        F.col("py.payment_status"),
        F.col("py.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("py.processed_at").cast("timestamp")
        ).alias("gold_updated_ts")
    )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        F.when(
            F.col("order_amount") > 0,
            F.col("paid_amount") / F.col("order_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "payment_sate",
        F.when(F.col("order_amount") == 0, "Invalid_order_amount")
        .when(F.col("payment_completion_ratio") == 0, "Unpaid")
        .when(F.col("payment_completion_ratio") == 1, "Paid")
        .when(F.col("payment_completion_ratio") < 1, "Partially_paid")
        .when(F.col("payment_completion_ratio") > 1, "Over_paid")
    )
    .withColumn("gold_updated_date", F.to_date(F.col("gold_updated_ts")))
    .withColumn("gold_run_id",F.lit(gold_run_id))
)

print("gold_delta_rows = ", gold_delta.count())
display(gold_delta) 
     

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(gold_delta, "novacart_db.gold_schema.orders_information", "order_id")
else:
    print("No new rows to insert in gold table")    

In [0]:
%sql
select * from novacart_db.gold_schema.orders_information;

In [0]:
if not spark.catalog.tableExists("novacart_db.gold_schema.orders_information_scd2"):
    spark.sql("""
              create table novacart_db.gold_schema.orders_information_scd2
              using delta
              as
              select *,
                    cast(null as timestamp) as valid_from_ts,
                    cast(null as timestamp) as valid_to_ts,
                    true as is_current
                    from novacart_db.gold_schema.orders_information
                    where 1 = 0
              """)

if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
              merge into novacart_db.gold_schema.orders_information_scd2 t
              using gold_delta_view s
              on t.order_id = s.order_id and t.is_current = true
              when matched and (
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price))
              then update set
                is_current = false,
                valid_to_ts = s.gold_updated_ts
                """)

    
    spark.sql("""
              insert into novacart_db.gold_schema.orders_information_scd2
              select s.*,
                    s.gold_updated_ts as valid_from_ts,
                    cast(null as timestamp) as valid_to_ts,
                    true as is_current
              from gold_delta_view s
              left join novacart_db.gold_schema.orders_information_scd2 t
              on s.order_id = t.order_id and t.is_current = true
              where t.order_id is null or (
                      not(t.order_status <=> s.order_status) or
                      not(t.order_amount <=> s.order_amount) or
                      not(t.paid_amount <=> s.paid_amount) or
                      not(t.payment_id <=> s.payment_id) or
                      not(t.category <=> s.category) or
                      not(t.product_name <=> s.product_name) or
                      not(t.product_price <=> s.product_price)
              )    
              """)
       
     

In [0]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta
        .select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
    )
    category_pref_delta = (
        spark.read.table("novacart_db.gold_schema.orders_information")
        .join(impacted_categories, "category", "inner")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("total_order"),
            F.sum(
                F.when(F.col("order_amount") > 0, F.col("order_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Gross_Merchandise_Value"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                .otherwise(F.lit(0.0))
            ).alias("Total_Paid_Amount"),
            F.avg(F.col("payment_completion_ratio")).alias("Average_Payment_Completion_Ratio"),
            (
                F.sum(F.when(F.col("Payment_status")== "Failed", 1).otherwise(0)) / F.count("*")
            ).alias("Payment_Failure_Rate")                                              
            )        
    )
    upsert_to_gold(category_pref_delta, "novacart_db.gold_schema.category_performance", "category")

In [0]:
%sql
select * from novacart_db.gold_schema.category_performance;

In [0]:
spark.sql("create volume if not exists novacart_db.gold_schema.gold_snapshots_vol")

In [0]:
latest_orders_path = (
    "/Volumes/novacart_db/gold_schema/gold_snapshots_vol/gold_latest/orders_information"
)

latest_category_path =(
    "/Volumes/novacart_db/gold_schema/gold_snapshots_vol/gold_latest/category_performance"
)

historical_orders_path = f"/Volumes/novacart_db/gold_schema/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"
historical_category_path = f"/Volumes/novacart_db/gold_schema/gold_snapshots_vol/gold_snapshots/category_performance/run_date={run_date_str}/run_ts={run_ts_str}"

spark.read.table("novacart_db.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)
spark.read.table("novacart_db.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(latest_category_path)

spark.read.table("novacart_db.gold_schema.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)
spark.read.table("novacart_db.gold_schema.category_performance").write.mode("overwrite").format("parquet").save(historical_category_path)

print("Latest Orders Path :", latest_orders_path)
print("Latest Category Path :", latest_category_path)
print("Historical Orders Path :", historical_orders_path)
print("Historical Category Path :", historical_category_path)

In [0]:
latest_silver_ts = silver_orders_current.agg(F.max("bronze_ingested_at").alias("mx")).collect()[0]['mx']

latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == latest_silver_ts)
    .agg(F.max("silver_run_id").alias("mx"))
    .collect()[0]['mx']
) if latest_silver_ts is not None else None

upser_gold_control("orders_information", latest_silver_run_id, latest_silver_ts, gold_delta.count())
display(spark.table("novacart_db.gold_schema.processing_control"))

In [0]:
%sql
select * from novacart_db.gold_schema.orders_information_scd2 where order_id = 200002;
     